# Meyaar Vision — E6 Paired Element Presence

هدف هذا النوتبوك: تدريب Florence-2 على **وجود/غياب عناصر الخريطة** باستخدام Controlled Error Injection.

بدل تصنيف الصورة مباشرة إلى 5 فئات، نسأل عن كل عنصر بشكل مستقل: `title`, `legend`, `scale`, `orientation`.

لكل خريطة يكون العنصر موجودًا فيها أصلًا:
- الصورة الأصلية → `yes`
- نسخة محذوف منها العنصر بالـinpainting → `no`

بهذا نستفيد من عدد أكبر من الخرائط بدل الاقتصار على الخرائط التي تحتوي العناصر الأربعة كلها.

**مهم:** التقسيم يتم على الخرائط الأصلية أولًا، ثم يتم توليد النسخ المحقونة داخل كل split لمنع Data Leakage. لا نستخدم Test إلا بعد اختيار أفضل مودل.

## 0. Setup

In [ ]:
!pip -q install -U \
    "transformers>=5.15.0,<5.16" \
    "peft>=0.17" "accelerate>=1.10" "datasets>=4.0" \
    "scikit-learn>=1.5" "opencv-python-headless>=4.10" \
    "pandas==2.2.3" "matplotlib>=3.9" "Pillow==11.3.0" \
    "torchao>=0.16.0"

> إذا طلب Colab إعادة تشغيل الـRuntime بعد التثبيت، أعيدي التشغيل ثم ابدئي من خلية الـimports.

In [ ]:
import os, gc, json, random, shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

## 1. Mount Drive + Local Workspace

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DRIVE_PROJECT = Path("/content/drive/MyDrive/Meyaar")
DRIVE_MAPS = DRIVE_PROJECT / "maps"
DRIVE_LABELS = DRIVE_PROJECT / "labels"
LOCAL_PROJECT = Path("/content/Meyaar")
MAPS_DIR = LOCAL_PROJECT / "maps"
LABELS_DIR = LOCAL_PROJECT / "labels"
PAIR_DIR = LOCAL_PROJECT / "e6_paired_presence"
CHECKPOINT_DIR = DRIVE_PROJECT / "checkpoints"
LOCAL_PROJECT.mkdir(parents=True, exist_ok=True)
PAIR_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive maps exists:", DRIVE_MAPS.exists())
print("Drive labels exists:", DRIVE_LABELS.exists())

In [ ]:
if not MAPS_DIR.exists():
    print("Copying maps..."); shutil.copytree(DRIVE_MAPS, MAPS_DIR)
if not LABELS_DIR.exists():
    print("Copying labels..."); shutil.copytree(DRIVE_LABELS, LABELS_DIR)
print("Maps:", len(list(MAPS_DIR.rglob("*.png"))))
print("JSON labels:", len(list(LABELS_DIR.rglob("*.json"))))

## 2. Read and Normalize Annotations

In [ ]:
ELEMENT_MAP = {
    "title":"title", "subtitle":"text",
    "legend-color":"legend", "legend-symbol":"legend", "legend-mixed":"legend",
    "scale-graphic":"scale", "scale-numeric":"scale",
    "orient-arrow":"orientation", "mapped_area":"mapped_area",
    "neat_line":"map_frame", "neat_line-in":"map_frame", "neat_line-out":"map_frame",
    "additional_notes":"text", "data_source":"text", "credit":"text",
    "inset":"inset", "orient-grids":"grid"
}
TARGET_ELEMENTS = ["title", "legend", "scale", "orientation"]

def annotation_to_bbox(obj):
    pts = obj.get("obj_points", [])
    if not pts: return None
    if obj.get("obj_type") == 1:
        p = pts[0]
        if not all(k in p for k in ("x","y","w","h")): return None
        x1,y1=float(p["x"]),float(p["y"]); x2,y2=x1+float(p["w"]),y1+float(p["h"])
    else:
        valid=[p for p in pts if "x" in p and "y" in p]
        if not valid: return None
        xs=[float(p["x"]) for p in valid]; ys=[float(p["y"]) for p in valid]
        x1,y1,x2,y2=min(xs),min(ys),max(xs),max(ys)
    return tuple(int(round(v)) for v in (x1,y1,x2,y2))

def clip_bbox(bbox,width,height):
    if bbox is None: return None
    x1,y1,x2,y2=bbox
    x1=max(0,min(width-1,x1)); y1=max(0,min(height-1,y1))
    x2=max(x1+1,min(width,x2)); y2=max(y1+1,min(height,y2))
    return (x1,y1,x2,y2)

def image_name_from_json(json_path):
    return json_path.name[:-5] if json_path.name.endswith(".json") else json_path.stem

def find_image(image_name):
    direct=MAPS_DIR/image_name
    if direct.exists(): return direct
    matches=list(MAPS_DIR.rglob(image_name)); return matches[0] if matches else None

def load_annotation_file(json_path):
    with open(json_path,"r",encoding="utf-8") as f: return json.load(f)

In [ ]:
json_files=sorted(LABELS_DIR.rglob("*.json")); paired=[]
for jp in json_files:
    img_path=find_image(image_name_from_json(jp))
    if img_path is not None: paired.append((img_path,jp))
print("JSON files:",len(json_files))
print("Matched image/JSON pairs:",len(paired),"/",len(json_files))

## 3. Build Original Map Index

In [ ]:
records=[]
for img_path,json_path in paired:
    try:
        with Image.open(img_path) as im: width,height=im.size
    except Exception as e:
        print("Skipping:",img_path,e); continue
    anns=load_annotation_file(json_path); elements=defaultdict(list)
    for obj in anns:
        raw_name=obj.get("f_name"); merged=ELEMENT_MAP.get(raw_name)
        bbox=clip_bbox(annotation_to_bbox(obj),width,height)
        if merged and bbox:
            elements[merged].append({"raw_name":raw_name,"bbox":bbox,"obj_type":obj.get("obj_type")})
    records.append({
        "map_id":img_path.name,"image_path":str(img_path),"json_path":str(json_path),
        "width":width,"height":height,"elements":dict(elements),
        "has_title":int(bool(elements.get("title"))),
        "has_legend":int(bool(elements.get("legend"))),
        "has_scale":int(bool(elements.get("scale"))),
        "has_orientation":int(bool(elements.get("orientation")))})
original_df=pd.DataFrame(records)
print("Original maps:",len(original_df))
for element in TARGET_ELEMENTS: print(f"{element:12s}:",int(original_df[f"has_{element}"].sum()))
display(original_df.head())

## 4. Split Original Maps Before Injection

In [ ]:
rng=np.random.default_rng(SEED); indices=np.arange(len(original_df)); rng.shuffle(indices)
n=len(indices); n_train=int(0.70*n); n_val=int(0.15*n)
train_idx=indices[:n_train]; val_idx=indices[n_train:n_train+n_val]; test_idx=indices[n_train+n_val:]
original_df["split"]=""
original_df.loc[train_idx,"split"]="train"; original_df.loc[val_idx,"split"]="val"; original_df.loc[test_idx,"split"]="test"
print(original_df["split"].value_counts())
presence_table=pd.DataFrame({e:original_df.groupby("split")[f"has_{e}"].sum() for e in TARGET_ELEMENTS})
display(presence_table)

## 5. Controlled Error Injection

In [ ]:
def expand_bbox(bbox,pad,width,height):
    x1,y1,x2,y2=bbox
    return (max(0,x1-pad),max(0,y1-pad),min(width,x2+pad),min(height,y2+pad))

def inpaint_region(pil_img,bbox,pad=3):
    arr=np.array(pil_img.convert("RGB")); h,w=arr.shape[:2]
    x1,y1,x2,y2=expand_bbox(bbox,pad,w,h)
    mask=np.zeros((h,w),dtype=np.uint8); mask[y1:y2,x1:x2]=255
    result=cv2.inpaint(arr,mask,3,cv2.INPAINT_TELEA)
    return Image.fromarray(result)

## 6. Build E6 Paired Presence Dataset

لكل عنصر موجود أصلًا: `original → yes` و `injected missing → no`.

In [ ]:
REGENERATE=True; rows=[]
for _,row in original_df.iterrows():
    split=row["split"]; map_id=row["map_id"]; elements=row["elements"]
    original_img=Image.open(row["image_path"]).convert("RGB")
    for element in TARGET_ELEMENTS:
        boxes=[item["bbox"] for item in elements.get(element,[])]
        if not boxes: continue
        rows.append({"source_map_id":map_id,"image_path":row["image_path"],"split":split,"element":element,"target":"yes","is_injected":False})
        out_dir=PAIR_DIR/split/element; out_dir.mkdir(parents=True,exist_ok=True)
        out_path=out_dir/f"{Path(map_id).stem}__missing_{element}.jpg"
        if REGENERATE or not out_path.exists():
            corrupted=original_img.copy()
            for bbox in boxes: corrupted=inpaint_region(corrupted,bbox)
            corrupted.save(out_path,quality=95)
        rows.append({"source_map_id":map_id,"image_path":str(out_path),"split":split,"element":element,"target":"no","is_injected":True})
paired_presence_df=pd.DataFrame(rows)
print("Total paired samples:",len(paired_presence_df))
print("
By split:"); print(paired_presence_df["split"].value_counts())
display(paired_presence_df.groupby(["split","element","target"]).size().rename("count").to_frame())

### Leakage Check

In [ ]:
split_counts=paired_presence_df.groupby("source_map_id")["split"].nunique()
assert split_counts.max()==1,"Data leakage detected!"
print("Leakage check passed ✅")
print("Train samples:",len(paired_presence_df[paired_presence_df["split"]=="train"]))

## 7. Visual Sanity Check

In [ ]:
sample_no=paired_presence_df[(paired_presence_df["split"]=="train")&(paired_presence_df["target"]=="no")].sample(4,random_state=SEED)
plt.figure(figsize=(12,12))
for i,(_,row) in enumerate(sample_no.iterrows(),1):
    img=Image.open(row["image_path"]).convert("RGB")
    ax=plt.subplot(2,2,i); ax.imshow(img); ax.set_title(f'{row["element"]} → {row["target"]}'); ax.axis("off")
plt.tight_layout(); plt.show()

## 8. Balance Train by Element × Target

In [ ]:
train_pairs=paired_presence_df[paired_presence_df["split"]=="train"].copy()
val_pairs=paired_presence_df[paired_presence_df["split"]=="val"].copy()
test_pairs=paired_presence_df[paired_presence_df["split"]=="test"].copy()
group_sizes=train_pairs.groupby(["element","target"]).size(); print("Before balancing:"); print(group_sizes)
MAX_GROUP=int(group_sizes.max()); balanced=[]
for (element,target),group in train_pairs.groupby(["element","target"]):
    balanced.append(group.sample(n=MAX_GROUP,replace=len(group)<MAX_GROUP,random_state=SEED))
train_balanced_df=pd.concat(balanced,ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
print("Balanced train samples:",len(train_balanced_df))
display(train_balanced_df.groupby(["element","target"]).size().rename("count").to_frame())

## 9. Load Florence-2

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
MODEL_ID="florence-community/Florence-2-base-ft"
processor=AutoProcessor.from_pretrained(MODEL_ID)
print("Processor loaded:",MODEL_ID)

## 10. Build Hugging Face Dataset

In [ ]:
train_ds=Dataset.from_pandas(train_balanced_df[["image_path","element","target","source_map_id"]],preserve_index=False)
val_ds=Dataset.from_pandas(val_pairs[["image_path","element","target","source_map_id"]],preserve_index=False)
print(train_ds); print(val_ds)

## 11. Prompt + Collate

In [ ]:
def make_presence_prompt(element):
    return f"Look at this map image. Is the {element} present? Answer exactly yes or no."

def collate_fn_presence(batch):
    images=[Image.open(x["image_path"]).convert("RGB") for x in batch]
    prompts=[make_presence_prompt(x["element"]) for x in batch]
    targets=[x["target"] for x in batch]
    inputs=processor(text=prompts,images=images,return_tensors="pt",padding=True)
    target_tokens=processor.tokenizer(targets,return_tensors="pt",padding=True)
    labels=target_tokens["input_ids"]; labels[labels==processor.tokenizer.pad_token_id]=-100
    inputs["labels"]=labels
    return inputs

## 12. Fresh LoRA Model — r=16

In [ ]:
from peft import LoraConfig,get_peft_model,TaskType
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
base_model=AutoModelForMultimodalLM.from_pretrained(MODEL_ID,torch_dtype=DTYPE).to(DEVICE)
lora_config=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,target_modules=["q_proj","v_proj"],task_type=TaskType.SEQ_2_SEQ_LM)
e6_model=get_peft_model(base_model,lora_config); e6_model.print_trainable_parameters()

## 13. Smoke Test — 2 Steps

In [ ]:
from transformers import TrainingArguments,Trainer
smoke_ds=train_ds.select(range(min(8,len(train_ds))))
smoke_args=TrainingArguments(output_dir="/content/e6_smoke",max_steps=2,per_device_train_batch_size=2,learning_rate=2e-4,logging_steps=1,fp16=torch.cuda.is_available(),report_to="none",remove_unused_columns=False,seed=SEED)
smoke_trainer=Trainer(model=e6_model,args=smoke_args,train_dataset=smoke_ds,data_collator=collate_fn_presence)
smoke_trainer.train()

إذا نجح الـSmoke Test، نعيد تحميل مودل Fresh قبل التدريب الكامل.

In [ ]:
e6_model.cpu(); del e6_model,base_model,smoke_trainer
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
base_model=AutoModelForMultimodalLM.from_pretrained(MODEL_ID,torch_dtype=DTYPE).to(DEVICE)
e6_model=get_peft_model(base_model,lora_config); e6_model.print_trainable_parameters()

## 14. Full E6 Training

In [ ]:
E6_OUTPUT_DIR=CHECKPOINT_DIR/"E6_paired_presence"
training_args=TrainingArguments(output_dir=str(E6_OUTPUT_DIR),num_train_epochs=3,per_device_train_batch_size=2,gradient_accumulation_steps=8,learning_rate=2e-4,logging_steps=10,save_strategy="epoch",save_total_limit=2,fp16=torch.cuda.is_available(),report_to="none",remove_unused_columns=False,seed=SEED)
trainer=Trainer(model=e6_model,args=training_args,train_dataset=train_ds,data_collator=collate_fn_presence)
train_output=trainer.train()

## 15. Save Final LoRA Adapter

In [ ]:
FINAL_SAVE=CHECKPOINT_DIR/"E6_paired_presence_final"
e6_model.save_pretrained(FINAL_SAVE); processor.save_pretrained(FINAL_SAVE)
print("Saved:",FINAL_SAVE)

## 16. Binary Validation

In [ ]:
def predict_presence(image_path,element):
    image=Image.open(image_path).convert("RGB"); prompt=make_presence_prompt(element)
    inputs=processor(text=prompt,images=image,return_tensors="pt")
    inputs={k:v.to(DEVICE) for k,v in inputs.items()}
    e6_model.eval()
    with torch.no_grad(): generated_ids=e6_model.generate(**inputs,max_new_tokens=5,do_sample=False,num_beams=1)
    output=processor.batch_decode(generated_ids,skip_special_tokens=True)[0].strip().lower()
    if output.startswith("yes"): return "yes"
    if output.startswith("no"): return "no"
    return "invalid"

In [ ]:
val_predictions=[]
for i,row in val_pairs.reset_index(drop=True).iterrows():
    val_predictions.append(predict_presence(row["image_path"],row["element"]))
    if (i+1)%50==0: print(f"{i+1}/{len(val_pairs)}")
binary_val_results=val_pairs.reset_index(drop=True).copy(); binary_val_results["prediction"]=val_predictions
print(binary_val_results["prediction"].value_counts())

In [ ]:
summary=[]
for element in TARGET_ELEMENTS:
    part=binary_val_results[binary_val_results["element"]==element]
    y_true,y_pred=part["target"],part["prediction"]
    acc=accuracy_score(y_true,y_pred)
    p,r,f1,_=precision_recall_fscore_support(y_true,y_pred,labels=["yes","no"],average="macro",zero_division=0)
    summary.append({"element":element,"n":len(part),"accuracy":acc,"macro_precision":p,"macro_recall":r,"macro_f1":f1})
element_metrics_df=pd.DataFrame(summary); display(element_metrics_df)
print("Overall Accuracy:",accuracy_score(binary_val_results["target"],binary_val_results["prediction"]))
p,r,f1,_=precision_recall_fscore_support(binary_val_results["target"],binary_val_results["prediction"],labels=["yes","no"],average="macro",zero_division=0)
print("Overall Macro Precision:",p); print("Overall Macro Recall:",r); print("Overall Macro F1:",f1)

## 17. Binary Confusion Matrix

In [ ]:
cm=confusion_matrix(binary_val_results["target"],binary_val_results["prediction"],labels=["yes","no"])
disp=ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=["yes","no"])
fig,ax=plt.subplots(figsize=(6,5)); disp.plot(ax=ax,values_format="d"); plt.title("E6 — Binary Presence Validation"); plt.tight_layout(); plt.show()

## 18. 5-Class Validation Benchmark

بعد ما نتأكد أن الـBinary Validation جيد، نستخدم فقط الخرائط الكاملة العناصر في Validation ونولد: `clean`, `missing_title`, `missing_legend`, `missing_scale`, `missing_orientation`.

In [ ]:
complete_val_df=original_df[(original_df["split"]=="val")&(original_df["has_title"]==1)&(original_df["has_legend"]==1)&(original_df["has_scale"]==1)&(original_df["has_orientation"]==1)].copy()
print("Complete validation source maps:",len(complete_val_df))

In [ ]:
FINAL_VAL_DIR=LOCAL_PROJECT/"e6_final_5class_val"; FINAL_VAL_DIR.mkdir(parents=True,exist_ok=True)
five_rows=[]
for _,row in complete_val_df.iterrows():
    img=Image.open(row["image_path"]).convert("RGB")
    five_rows.append({"source_map_id":row["map_id"],"image_path":row["image_path"],"error_type":"clean"})
    for element in TARGET_ELEMENTS:
        corrupted=img.copy(); boxes=[x["bbox"] for x in row["elements"][element]]
        for bbox in boxes: corrupted=inpaint_region(corrupted,bbox)
        out_path=FINAL_VAL_DIR/f"{Path(row['map_id']).stem}__missing_{element}.jpg"; corrupted.save(out_path,quality=95)
        five_rows.append({"source_map_id":row["map_id"],"image_path":str(out_path),"error_type":f"missing_{element}"})
five_class_val_df=pd.DataFrame(five_rows)
print("5-class validation samples:",len(five_class_val_df)); print(five_class_val_df["error_type"].value_counts())

In [ ]:
FINAL_LABELS=["clean","missing_title","missing_legend","missing_scale","missing_orientation"]
def predict_five_class(image_path):
    answers={e:predict_presence(image_path,e) for e in TARGET_ELEMENTS}
    if "invalid" in answers.values(): return "invalid",answers
    missing=[e for e,a in answers.items() if a=="no"]
    if len(missing)==0: return "clean",answers
    if len(missing)==1: return f"missing_{missing[0]}",answers
    return "multiple_missing",answers

In [ ]:
five_preds=[]; answer_records=[]
for i,row in five_class_val_df.iterrows():
    pred,answers=predict_five_class(row["image_path"]); five_preds.append(pred); answer_records.append(answers)
    if (i+1)%10==0: print(f"{i+1}/{len(five_class_val_df)}")
five_class_results=five_class_val_df.copy(); five_class_results["prediction"]=five_preds; five_class_results["answers"]=answer_records
print(five_class_results["prediction"].value_counts())

In [ ]:
y_true=five_class_results["error_type"]; y_pred=five_class_results["prediction"]
accuracy=accuracy_score(y_true,y_pred)
p,r,f1,_=precision_recall_fscore_support(y_true,y_pred,labels=FINAL_LABELS,average="macro",zero_division=0)
print("E6 Final 5-Class Validation"); print("---------------------------")
print(f"Accuracy        : {accuracy:.4f}"); print(f"Macro Precision : {p:.4f}"); print(f"Macro Recall    : {r:.4f}"); print(f"Macro F1        : {f1:.4f}")
print("
Classification Report:
"); print(classification_report(y_true,y_pred,labels=FINAL_LABELS,zero_division=0))

In [ ]:
cm=confusion_matrix(y_true,y_pred,labels=FINAL_LABELS)
disp=ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=FINAL_LABELS)
fig,ax=plt.subplots(figsize=(9,7)); disp.plot(ax=ax,xticks_rotation=45,values_format="d"); plt.title("E6 — Final Missing Elements Validation"); plt.tight_layout(); plt.show()

## 19. Final Test — لا تشغليه الآن

اتركي الـTest untouched إلى أن نقرر أن E6 هو أفضل إعداد بناءً على Validation.